# Create canonical FEA analysis datasets

This notebook creates the canonical FEA tables used by the analyses in
`6_discussion/fea_analysis`.

It produces six CSV files:

- `data/dfea_train.csv`
- `data/dfea_validation.csv`
- `data/dfea_test.csv`
- `data/sfea_train.csv`
- `data/sfea_validation.csv`
- `data/sfea_test.csv`

The DFEA tables use one row per **reenactment × chronological timestep**.
The SFEA tables use one row per **reenactment**.

The schema is intentionally compatible with the existing repository outputs:

- `6_discussion/significance-tests/dynamic-significance-tests/dynamic_test_predictions.csv`
- `5_dynamic_facial_expression_recognition/5_2_fea_sequence_based_fer/5_2_4_sequence_order_ablation/trajectories/fea_prediction_trajectories.csv`
- `6_discussion/significance-tests/static-significance-tests/static_test_predictions.csv`

The notebook also verifies that SFEA and the reference observation inside DFEA agree for the same reenactment.

> Run this notebook from `emohevrdb-dfer/6_discussion/fea_analysis/`.


## Paths

The defaults follow the dataset locations used by the existing prediction notebooks.

If your local mounts differ, edit only the candidate lists below. The notebook selects the first existing path.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

ANALYSIS_DIR = Path.cwd()
OUTPUT_DIR = ANALYSIS_DIR / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Existing dynamic notebooks use /workspace/datasets/emoji-hero-vr-db-dfea-as-csv.
DFEA_ROOT_CANDIDATES = [
    Path("/workspace/datasets/emoji-hero-vr-db-dfea-as-csv"),
    Path("/workspace/datasets/emohevrdb/emoji-hero-vr-db-dfea-as-csv"),
]

# Existing static prediction notebooks use
# /workspace/datasets/emohevrdb/emoji-hero-vr-db-sfea-as-csv.
SFEA_ROOT_CANDIDATES = [
    Path("/workspace/datasets/emohevrdb/emoji-hero-vr-db-sfea-as-csv"),
    Path("/workspace/datasets/emoji-hero-vr-db-sfea-as-csv"),
]

def first_existing(candidates, description):
    for path in candidates:
        if path.exists():
            print(f"{description}: {path}")
            return path

    candidates_text = "\n".join(f"  - {path}" for path in candidates)
    raise FileNotFoundError(
        f"Could not find {description}. Checked:\n{candidates_text}\n"
        "Edit the candidate paths in this cell if your dataset mount differs."
    )

DFEA_ROOT = first_existing(DFEA_ROOT_CANDIDATES, "DFEA source")
SFEA_ROOT = first_existing(SFEA_ROOT_CANDIDATES, "SFEA source")

SPLIT_FILES = {
    "train": "training_set.csv",
    "validation": "validation_set.csv",
    "test": "test_set.csv",
}

EXPECTED_REENACTMENTS = {
    "train": 964,
    "validation": 385,
    "test": 378,
}

SEQUENCE_LENGTH = 30

print("Output:", OUTPUT_DIR.resolve())


DFEA source: /workspace/datasets/emoji-hero-vr-db-dfea-as-csv
SFEA source: /workspace/datasets/emoji-hero-vr-db-sfea-as-csv
Output: /workspace/repos/emohevrdb-dfer/6_discussion/fea_analysis/data


## Canonical labels and FEA columns

The feature order below is the exact 63-channel order used by the saved static and dynamic prediction notebooks.


In [2]:
ID_TO_EMOTION = {
    0: "Anger",
    1: "Disgust",
    2: "Fear",
    3: "Happiness",
    4: "Neutral",
    5: "Sadness",
    6: "Surprise",
}

FEA_COLUMNS = [
    "BrowLowererL",
    "BrowLowererR",
    "CheekPuffL",
    "CheekPuffR",
    "CheekRaiserL",
    "CheekRaiserR",
    "CheekSuckL",
    "CheekSuckR",
    "ChinRaiserB",
    "ChinRaiserT",
    "DimplerL",
    "DimplerR",
    "EyesClosedL",
    "EyesClosedR",
    "EyesLookDownL",
    "EyesLookDownR",
    "EyesLookLeftL",
    "EyesLookLeftR",
    "EyesLookRightL",
    "EyesLookRightR",
    "EyesLookUpL",
    "EyesLookUpR",
    "InnerBrowRaiserL",
    "InnerBrowRaiserR",
    "JawDrop",
    "JawSidewaysLeft",
    "JawSidewaysRight",
    "JawThrust",
    "LidTightenerL",
    "LidTightenerR",
    "LipCornerDepressorL",
    "LipCornerDepressorR",
    "LipCornerPullerL",
    "LipCornerPullerR",
    "LipFunnelerLB",
    "LipFunnelerLT",
    "LipFunnelerRB",
    "LipFunnelerRT",
    "LipPressorL",
    "LipPressorR",
    "LipPuckerL",
    "LipPuckerR",
    "LipStretcherL",
    "LipStretcherR",
    "LipSuckLB",
    "LipSuckLT",
    "LipSuckRB",
    "LipSuckRT",
    "LipTightenerL",
    "LipTightenerR",
    "LipsToward",
    "LowerLipDepressorL",
    "LowerLipDepressorR",
    "MouthLeft",
    "MouthRight",
    "NoseWrinklerL",
    "NoseWrinklerR",
    "OuterBrowRaiserL",
    "OuterBrowRaiserR",
    "UpperLidRaiserL",
    "UpperLidRaiserR",
    "UpperLipRaiserL",
    "UpperLipRaiserR",
]

assert len(FEA_COLUMNS) == 63
assert len(FEA_COLUMNS) == len(set(FEA_COLUMNS))


## Helper functions

`reenactment_id` is the canonical project identifier. Source DFEA calls the same identifier `sequence_id`, while source SFEA calls it `file_id`.


In [3]:
def parse_reenactment_id(reenactment_id: str) -> dict:
    parts = str(reenactment_id).split("-")

    if len(parts) != 6:
        raise ValueError(
            f"Unexpected reenactment ID format: {reenactment_id!r}. "
            "Expected <timestamp>-<set>-<participant>-<level>-<emoji>-<emotion>."
        )

    sequence_timestamp, set_id, participant_id, level_id, emoji_id, emotion_id = parts

    return {
        "sequence_timestamp": int(sequence_timestamp),
        "set_id": int(set_id),
        "participant_id": int(participant_id),
        "level_id": int(level_id),
        "emoji_id": int(emoji_id),
        "emotion_id_from_id": int(emotion_id),
    }


def validate_source_schema(df: pd.DataFrame, id_column: str, source_name: str) -> None:
    expected_columns = [id_column, "timestamp", *FEA_COLUMNS, "Label"]

    assert df.columns.tolist() == expected_columns, (
        f"{source_name}: unexpected columns/order.\n"
        f"Expected: {expected_columns}\n"
        f"Actual:   {df.columns.tolist()}"
    )

    assert df[id_column].notna().all(), f"{source_name}: missing IDs"
    assert df["timestamp"].notna().all(), f"{source_name}: missing timestamps"
    assert df["Label"].notna().all(), f"{source_name}: missing labels"
    assert df[FEA_COLUMNS].notna().all().all(), f"{source_name}: missing FEA values"

    values = df[FEA_COLUMNS].to_numpy(dtype=float)
    assert np.isfinite(values).all(), f"{source_name}: non-finite FEA values"

    minimum = float(values.min())
    maximum = float(values.max())
    assert minimum >= -1e-8 and maximum <= 1.0 + 1e-8, (
        f"{source_name}: FEA range outside expected [0, 1]: "
        f"min={minimum}, max={maximum}"
    )


def add_reenactment_metadata(df: pd.DataFrame) -> pd.DataFrame:
    metadata = pd.DataFrame(
        [parse_reenactment_id(value) for value in df["reenactment_id"]],
        index=df.index,
    )

    return pd.concat([df, metadata], axis=1)


## Build canonical DFEA splits

Each source sequence is sorted by `timestamp`, then assigned `timestep = 1, ..., 30`.

The output key is `(reenactment_id, timestep)`.


In [4]:
def build_dfea_split(split: str) -> pd.DataFrame:
    source_path = DFEA_ROOT / SPLIT_FILES[split]
    raw = pd.read_csv(source_path)

    validate_source_schema(
        raw,
        id_column="sequence_id",
        source_name=f"DFEA {split}",
    )

    raw = raw.rename(
        columns={
            "sequence_id": "reenactment_id",
            "timestamp": "observation_timestamp",
            "Label": "true_label_id",
        }
    )

    rows = []

    for reenactment_id, group in raw.groupby("reenactment_id", sort=True):
        group = group.sort_values("observation_timestamp").reset_index(drop=True)

        assert len(group) == SEQUENCE_LENGTH
        assert group["true_label_id"].nunique() == 1
        assert group["observation_timestamp"].is_unique

        parsed = parse_reenactment_id(reenactment_id)
        true_label_id = int(group["true_label_id"].iloc[0])

        assert parsed["emotion_id_from_id"] == true_label_id

        group = group.copy()
        group.insert(0, "timestep", np.arange(1, SEQUENCE_LENGTH + 1, dtype=int))
        group.insert(0, "split", split)

        for key in [
            "sequence_timestamp",
            "set_id",
            "participant_id",
            "level_id",
            "emoji_id",
        ]:
            group[key] = parsed[key]

        group["true_label_id"] = true_label_id
        group["true_label"] = ID_TO_EMOTION[true_label_id]
        group["is_reference_observation"] = (
            group["observation_timestamp"] == parsed["sequence_timestamp"]
        )

        reference_count = int(group["is_reference_observation"].sum())
        assert reference_count == 1, (
            f"{split} / {reenactment_id}: expected exactly one observation "
            f"at sequence_timestamp={parsed['sequence_timestamp']}, "
            f"found {reference_count}."
        )

        rows.append(group)

    result = pd.concat(rows, ignore_index=True)

    column_order = [
        "reenactment_id",
        "split",
        "sequence_timestamp",
        "set_id",
        "participant_id",
        "level_id",
        "emoji_id",
        "timestep",
        "observation_timestamp",
        "is_reference_observation",
        "true_label_id",
        "true_label",
        *FEA_COLUMNS,
    ]

    result = result[column_order].sort_values(
        ["reenactment_id", "timestep"]
    ).reset_index(drop=True)

    expected_n = EXPECTED_REENACTMENTS[split]

    assert result["reenactment_id"].nunique() == expected_n
    assert len(result) == expected_n * SEQUENCE_LENGTH
    assert not result.duplicated(["reenactment_id", "timestep"]).any()
    assert result.groupby("reenactment_id").size().eq(SEQUENCE_LENGTH).all()
    assert result.groupby("reenactment_id")["timestep"].nunique().eq(SEQUENCE_LENGTH).all()
    assert result.groupby("reenactment_id")["is_reference_observation"].sum().eq(1).all()

    return result


dfea = {
    split: build_dfea_split(split)
    for split in SPLIT_FILES
}

for split, df in dfea.items():
    print(
        f"DFEA {split:10s}: "
        f"{len(df):>6,} rows, "
        f"{df['reenactment_id'].nunique():>3} reenactments, "
        f"{df['participant_id'].nunique():>2} participants"
    )

display(dfea["test"].head())


DFEA train     : 28,920 rows, 964 reenactments, 20 participants
DFEA validation: 11,550 rows, 385 reenactments,  8 participants
DFEA test      : 11,340 rows, 378 reenactments,  8 participants


,reenactment_id,split,sequence_timestamp,set_id,participant_id,level_id,emoji_id,timestep,observation_timestamp,is_reference_observation,...,MouthLeft,MouthRight,NoseWrinklerL,NoseWrinklerR,OuterBrowRaiserL,OuterBrowRaiserR,UpperLidRaiserL,UpperLidRaiserR,UpperLipRaiserL,UpperLipRaiserR
0,1700478995850-2-1-1-0-0,test,1700478995850,2,1,1,0,1,1700478994882,False,...,8.933511e-08,6.212495e-08,2.818527e-11,4.151673e-11,2.690706e-13,0.000217,1.401298e-45,1.401298e-45,0.015593,0.010760
1,1700478995850-2-1-1-0-0,test,1700478995850,2,1,1,0,2,1700478994922,False,...,6.283343e-08,4.365342e-08,1.985715e-11,2.925454e-11,2.224745e-13,0.000179,1.401298e-45,1.401298e-45,0.014098,0.010794
2,1700478995850-2-1-1-0-0,test,1700478995850,2,1,1,0,3,1700478994949,False,...,4.420946e-08,3.069010e-08,1.399077e-11,2.061487e-11,1.839484e-13,0.000148,1.401298e-45,1.401298e-45,0.022011,0.010817
3,1700478995850-2-1-1-0-0,test,1700478995850,2,1,1,0,4,1700478994977,False,...,3.111433e-08,2.158532e-08,9.857880e-12,1.452693e-11,1.520928e-13,0.000218,1.401298e-45,1.401298e-45,0.017628,0.010832
4,1700478995850-2-1-1-0-0,test,1700478995850,2,1,1,0,5,1700478995018,False,...,2.190368e-08,1.518726e-08,6.946256e-12,1.023726e-11,1.257550e-13,0.000180,1.401298e-45,1.401298e-45,0.015070,0.010843


## Build canonical SFEA splits

SFEA uses one row per reenactment and the same canonical identifier and class metadata as DFEA.


In [5]:
def build_sfea_split(split: str) -> pd.DataFrame:
    source_path = SFEA_ROOT / SPLIT_FILES[split]
    raw = pd.read_csv(source_path)

    validate_source_schema(
        raw,
        id_column="file_id",
        source_name=f"SFEA {split}",
    )

    raw = raw.rename(
        columns={
            "file_id": "reenactment_id",
            "timestamp": "observation_timestamp",
            "Label": "true_label_id",
        }
    )

    result = add_reenactment_metadata(raw)
    result.insert(1, "split", split)

    assert result["reenactment_id"].is_unique

    result["true_label_id"] = result["true_label_id"].astype(int)
    result["true_label"] = result["true_label_id"].map(ID_TO_EMOTION)

    assert result["true_label"].notna().all()
    assert (
        result["emotion_id_from_id"].astype(int)
        == result["true_label_id"]
    ).all()

    result = result.drop(columns="emotion_id_from_id")

    column_order = [
        "reenactment_id",
        "split",
        "sequence_timestamp",
        "set_id",
        "participant_id",
        "level_id",
        "emoji_id",
        "observation_timestamp",
        "true_label_id",
        "true_label",
        *FEA_COLUMNS,
    ]

    result = result[column_order].sort_values("reenactment_id").reset_index(drop=True)

    expected_n = EXPECTED_REENACTMENTS[split]
    assert len(result) == expected_n
    assert result["reenactment_id"].nunique() == expected_n

    return result


sfea = {
    split: build_sfea_split(split)
    for split in SPLIT_FILES
}

for split, df in sfea.items():
    print(
        f"SFEA {split:10s}: "
        f"{len(df):>3} rows, "
        f"{df['participant_id'].nunique():>2} participants"
    )

display(sfea["test"].head())


SFEA train     : 964 rows, 20 participants
SFEA validation: 385 rows,  8 participants
SFEA test      : 378 rows,  8 participants


,reenactment_id,split,sequence_timestamp,set_id,participant_id,level_id,emoji_id,observation_timestamp,true_label_id,true_label,...,MouthLeft,MouthRight,NoseWrinklerL,NoseWrinklerR,OuterBrowRaiserL,OuterBrowRaiserR,UpperLidRaiserL,UpperLidRaiserR,UpperLipRaiserL,UpperLipRaiserR
0,1700478995850-2-1-1-0-0,test,1700478995850,2,1,1,0,1700478995850,0,Anger,...,3.463278e-12,2.395155e-12,2.410580e-03,2.882660e-03,1.083685e-15,0.000062,1.401298e-45,1.401298e-45,4.814688e-03,2.418802e-03
1,1700478998549-2-1-1-1-5,test,1700478998549,2,1,1,1,1700478998549,5,Sadness,...,1.397256e-14,1.179313e-24,3.015459e-08,3.433736e-06,7.538587e-05,0.022539,1.401298e-45,1.401298e-45,3.242157e-10,2.119901e-11
2,1700479001137-2-1-1-2-3,test,1700479001137,2,1,1,2,1700479001137,3,Happiness,...,1.965327e-26,1.057651e-03,4.234153e-20,4.582445e-18,8.560760e-04,0.001896,1.401298e-45,1.401298e-45,4.317500e-01,4.569992e-01
3,1700479004312-2-1-1-3-0,test,1700479004312,2,1,1,3,1700479004312,0,Anger,...,5.898144e-03,2.944601e-07,2.812969e-14,2.812969e-14,1.442060e-11,0.001512,1.401298e-45,1.401298e-45,6.511594e-03,8.534960e-03
4,1700479005401-2-1-1-4-0,test,1700479005401,2,1,1,4,1700479005401,0,Anger,...,2.908783e-07,9.621919e-03,2.722222e-19,2.722222e-19,2.714441e-14,0.000358,1.401298e-45,1.401298e-45,1.212497e-02,1.086563e-02


## Cross-check SFEA against the DFEA reference observation

For each reenactment, DFEA must contain exactly one observation whose timestamp equals the sequence/reference timestamp.

This cell verifies that this DFEA reference observation has the same label and 63 FEA values as the corresponding SFEA sample.


In [6]:
for split in SPLIT_FILES:
    dfea_reference = (
        dfea[split]
        .loc[dfea[split]["is_reference_observation"]]
        .copy()
        .sort_values("reenactment_id")
        .reset_index(drop=True)
    )

    sfea_split = (
        sfea[split]
        .sort_values("reenactment_id")
        .reset_index(drop=True)
    )

    assert dfea_reference["reenactment_id"].tolist() == sfea_split["reenactment_id"].tolist()
    assert np.array_equal(
        dfea_reference["true_label_id"].to_numpy(),
        sfea_split["true_label_id"].to_numpy(),
    )

    dfea_values = dfea_reference[FEA_COLUMNS].to_numpy(dtype=float)
    sfea_values = sfea_split[FEA_COLUMNS].to_numpy(dtype=float)

    max_abs_difference = float(np.max(np.abs(dfea_values - sfea_values)))

    assert np.allclose(
        dfea_values,
        sfea_values,
        rtol=0.0,
        atol=1e-8,
    ), (
        f"{split}: DFEA reference observations and SFEA values differ. "
        f"Maximum absolute difference: {max_abs_difference}"
    )

    print(
        f"{split:10s}: SFEA == DFEA reference observation "
        f"for {len(sfea_split)} reenactments "
        f"(max abs diff = {max_abs_difference:.3g})"
    )


train     : SFEA == DFEA reference observation for 964 reenactments (max abs diff = 0)
validation: SFEA == DFEA reference observation for 385 reenactments (max abs diff = 0)
test      : SFEA == DFEA reference observation for 378 reenactments (max abs diff = 0)


## Validate participant-disjoint splits

The same split logic must be preserved in both SFEA and DFEA.


In [7]:
participant_sets = {
    split: set(dfea[split]["participant_id"].unique())
    for split in SPLIT_FILES
}

assert participant_sets["train"].isdisjoint(participant_sets["validation"])
assert participant_sets["train"].isdisjoint(participant_sets["test"])
assert participant_sets["validation"].isdisjoint(participant_sets["test"])

for split in SPLIT_FILES:
    assert participant_sets[split] == set(sfea[split]["participant_id"].unique())

for split, participants in participant_sets.items():
    print(f"{split:10s}: {len(participants):>2} participants -> {sorted(participants)}")


train     : 20 participants -> [3, 4, 11, 12, 16, 17, 19, 20, 21, 22, 24, 25, 26, 28, 30, 31, 33, 34, 35, 37]
validation:  8 participants -> [2, 5, 6, 9, 14, 29, 32, 36]
test      :  8 participants -> [1, 8, 10, 13, 15, 18, 23, 27]


## Compatibility checks against existing prediction artifacts

These checks are optional in the sense that the notebook can create the canonical datasets without the prediction files. When the corresponding repository outputs exist, the notebook verifies the expected joins.

- Dynamic final predictions: `reenactment_id`
- FEA prediction trajectories: `(reenactment_id, timestep)`
- Static predictions: `reenactment_id`


In [8]:
REPO_ROOT = (ANALYSIS_DIR / "../..").resolve()

DYNAMIC_PREDICTIONS_PATH = (
    REPO_ROOT
    / "6_discussion"
    / "significance-tests"
    / "dynamic-significance-tests"
    / "dynamic_test_predictions.csv"
)

FEA_TRAJECTORIES_PATH = (
    REPO_ROOT
    / "5_dynamic_facial_expression_recognition"
    / "5_2_fea_sequence_based_fer"
    / "5_2_4_sequence_order_ablation"
    / "trajectories"
    / "fea_prediction_trajectories.csv"
)

STATIC_PREDICTIONS_PATH = (
    REPO_ROOT
    / "6_discussion"
    / "significance-tests"
    / "static-significance-tests"
    / "static_test_predictions.csv"
)


def check_dynamic_predictions():
    if not DYNAMIC_PREDICTIONS_PATH.exists():
        print("SKIP dynamic final predictions:", DYNAMIC_PREDICTIONS_PATH)
        return

    predictions = pd.read_csv(DYNAMIC_PREDICTIONS_PATH)

    assert predictions["reenactment_id"].nunique() == EXPECTED_REENACTMENTS["test"]
    assert predictions.groupby("reenactment_id").size().eq(2).all()

    canonical_ids = set(dfea["test"]["reenactment_id"].unique())
    prediction_ids = set(predictions["reenactment_id"].unique())
    assert canonical_ids == prediction_ids

    label_table = (
        predictions[["reenactment_id", "true_label_id", "true_label"]]
        .drop_duplicates()
    )
    assert label_table["reenactment_id"].is_unique

    canonical_labels = (
        dfea["test"][["reenactment_id", "true_label_id", "true_label"]]
        .drop_duplicates()
    )

    merged = canonical_labels.merge(
        label_table,
        on="reenactment_id",
        how="left",
        validate="one_to_one",
        suffixes=("_canonical", "_prediction"),
    )

    assert (
        merged["true_label_id_canonical"]
        == merged["true_label_id_prediction"]
    ).all()
    assert (
        merged["true_label_canonical"]
        == merged["true_label_prediction"]
    ).all()

    if "fea_pred_id" in predictions.columns:
        assert predictions.groupby("reenactment_id")["fea_pred_id"].nunique().eq(1).all()

    print("PASS dynamic final predictions:", DYNAMIC_PREDICTIONS_PATH)


def check_fea_trajectories():
    if not FEA_TRAJECTORIES_PATH.exists():
        print("SKIP FEA prediction trajectories:", FEA_TRAJECTORIES_PATH)
        return

    trajectories = pd.read_csv(FEA_TRAJECTORIES_PATH).rename(
        columns={
            "sequence_id": "reenactment_id",
            "true_class_id": "true_label_id",
            "true_class": "true_label",
        }
    )

    assert len(trajectories) == EXPECTED_REENACTMENTS["test"] * SEQUENCE_LENGTH
    assert not trajectories.duplicated(["reenactment_id", "timestep"]).any()

    canonical = dfea["test"][
        [
            "reenactment_id",
            "timestep",
            "observation_timestamp",
            "true_label_id",
            "true_label",
        ]
    ]

    merged = canonical.merge(
        trajectories[
            [
                "reenactment_id",
                "timestep",
                "observation_timestamp",
                "true_label_id",
                "true_label",
            ]
        ],
        on=["reenactment_id", "timestep"],
        how="left",
        validate="one_to_one",
        suffixes=("_canonical", "_trajectory"),
    )

    assert len(merged) == len(canonical)
    assert merged["observation_timestamp_trajectory"].notna().all()
    assert np.array_equal(
        merged["observation_timestamp_canonical"].to_numpy(),
        merged["observation_timestamp_trajectory"].to_numpy(),
    )
    assert np.array_equal(
        merged["true_label_id_canonical"].to_numpy(),
        merged["true_label_id_trajectory"].to_numpy(),
    )
    assert (
        merged["true_label_canonical"]
        == merged["true_label_trajectory"]
    ).all()

    print("PASS FEA prediction trajectories:", FEA_TRAJECTORIES_PATH)


def check_static_predictions():
    if not STATIC_PREDICTIONS_PATH.exists():
        print("SKIP static predictions:", STATIC_PREDICTIONS_PATH)
        return

    predictions = pd.read_csv(STATIC_PREDICTIONS_PATH)

    assert predictions["reenactment_id"].nunique() == EXPECTED_REENACTMENTS["test"]
    assert predictions.groupby("reenactment_id").size().eq(2).all()

    canonical_ids = set(sfea["test"]["reenactment_id"])
    prediction_ids = set(predictions["reenactment_id"])
    assert canonical_ids == prediction_ids

    prediction_labels = (
        predictions[["reenactment_id", "true_label_id", "true_label"]]
        .drop_duplicates()
    )
    assert prediction_labels["reenactment_id"].is_unique

    canonical_labels = sfea["test"][
        ["reenactment_id", "true_label_id", "true_label"]
    ]

    merged = canonical_labels.merge(
        prediction_labels,
        on="reenactment_id",
        how="left",
        validate="one_to_one",
        suffixes=("_canonical", "_prediction"),
    )

    assert (
        merged["true_label_id_canonical"]
        == merged["true_label_id_prediction"]
    ).all()
    assert (
        merged["true_label_canonical"]
        == merged["true_label_prediction"]
    ).all()

    print("PASS static predictions:", STATIC_PREDICTIONS_PATH)


check_dynamic_predictions()
check_fea_trajectories()
check_static_predictions()


PASS dynamic final predictions: /workspace/repos/emohevrdb-dfer/6_discussion/significance-tests/dynamic-significance-tests/dynamic_test_predictions.csv
PASS FEA prediction trajectories: /workspace/repos/emohevrdb-dfer/5_dynamic_facial_expression_recognition/5_2_fea_sequence_based_fer/5_2_4_sequence_order_ablation/trajectories/fea_prediction_trajectories.csv
PASS static predictions: /workspace/repos/emohevrdb-dfer/6_discussion/significance-tests/static-significance-tests/static_test_predictions.csv


## Export canonical CSV files

The split-specific files deliberately retain the `split` column. This makes provenance explicit when tables are later concatenated.


In [9]:
output_paths = {}

for split, df in dfea.items():
    path = OUTPUT_DIR / f"dfea_{split}.csv"
    df.to_csv(path, index=False)
    output_paths[f"dfea_{split}"] = path

for split, df in sfea.items():
    path = OUTPUT_DIR / f"sfea_{split}.csv"
    df.to_csv(path, index=False)
    output_paths[f"sfea_{split}"] = path

for name, path in output_paths.items():
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{name:18s} -> {path} ({size_mb:.2f} MB)")


dfea_train         -> /workspace/repos/emohevrdb-dfer/6_discussion/fea_analysis/data/dfea_train.csv (35.69 MB)
dfea_validation    -> /workspace/repos/emohevrdb-dfer/6_discussion/fea_analysis/data/dfea_validation.csv (14.17 MB)
dfea_test          -> /workspace/repos/emohevrdb-dfer/6_discussion/fea_analysis/data/dfea_test.csv (13.92 MB)
sfea_train         -> /workspace/repos/emohevrdb-dfer/6_discussion/fea_analysis/data/sfea_train.csv (1.17 MB)
sfea_validation    -> /workspace/repos/emohevrdb-dfer/6_discussion/fea_analysis/data/sfea_validation.csv (0.47 MB)
sfea_test          -> /workspace/repos/emohevrdb-dfer/6_discussion/fea_analysis/data/sfea_test.csv (0.46 MB)


## Save a compact validation summary

This file is useful for quickly confirming which source files and dataset counts produced the canonical tables.


In [10]:
summary = {
    "schema_version": "1.0.0",
    "sequence_length": SEQUENCE_LENGTH,
    "fea_feature_count": len(FEA_COLUMNS),
    "dfea_source": str(DFEA_ROOT),
    "sfea_source": str(SFEA_ROOT),
    "splits": {},
    "compatibility_files": {
        "dynamic_test_predictions": str(DYNAMIC_PREDICTIONS_PATH),
        "fea_prediction_trajectories": str(FEA_TRAJECTORIES_PATH),
        "static_test_predictions": str(STATIC_PREDICTIONS_PATH),
    },
}

for split in SPLIT_FILES:
    reference_steps = (
        dfea[split]
        .loc[dfea[split]["is_reference_observation"], "timestep"]
        .value_counts()
        .sort_index()
    )

    summary["splits"][split] = {
        "reenactments": int(dfea[split]["reenactment_id"].nunique()),
        "dfea_rows": int(len(dfea[split])),
        "sfea_rows": int(len(sfea[split])),
        "participants": int(dfea[split]["participant_id"].nunique()),
        "reference_timestep_counts": {
            str(int(step)): int(count)
            for step, count in reference_steps.items()
        },
    }

summary_path = OUTPUT_DIR / "canonical_dataset_validation.json"
summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print(summary_path)
print(json.dumps(summary, indent=2))


/workspace/repos/emohevrdb-dfer/6_discussion/fea_analysis/data/canonical_dataset_validation.json
{
  "schema_version": "1.0.0",
  "sequence_length": 30,
  "fea_feature_count": 63,
  "dfea_source": "/workspace/datasets/emoji-hero-vr-db-dfea-as-csv",
  "sfea_source": "/workspace/datasets/emoji-hero-vr-db-sfea-as-csv",
  "splits": {
    "train": {
      "reenactments": 964,
      "dfea_rows": 28920,
      "sfea_rows": 964,
      "participants": 20,
      "reference_timestep_counts": {
        "16": 21,
        "17": 13,
        "18": 17,
        "19": 13,
        "20": 12,
        "21": 21,
        "22": 9,
        "23": 14,
        "24": 12,
        "25": 16,
        "26": 15,
        "27": 18,
        "28": 28,
        "29": 25,
        "30": 730
      }
    },
    "validation": {
      "reenactments": 385,
      "dfea_rows": 11550,
      "sfea_rows": 385,
      "participants": 8,
      "reference_timestep_counts": {
        "16": 4,
        "17": 7,
        "18": 2,
        "19": 5,
  

## Typical downstream loading

Dataset/signal characterization may concatenate all three participant-disjoint splits:

```python
dfea_all = pd.concat(
    [
        pd.read_csv("data/dfea_train.csv"),
        pd.read_csv("data/dfea_validation.csv"),
        pd.read_csv("data/dfea_test.csv"),
    ],
    ignore_index=True,
)
```

Final model evaluation, prediction-error analysis, prediction trajectories, and model-reliance analyses should use the held-out test split only:

```python
dfea_test = pd.read_csv("data/dfea_test.csv")
```

To attach the existing FEA prediction trajectories:

```python
trajectory_df = pd.read_csv(FEA_TRAJECTORIES_PATH).rename(
    columns={"sequence_id": "reenactment_id"}
)

dfea_test_with_predictions = dfea_test.merge(
    trajectory_df,
    on=["reenactment_id", "timestep"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_trajectory"),
)
```
